# 01. Exploratory Data Analysis & Quality Audit
### RoadGuardian AI: Video-Based Road Accident Detection and Incident Management

**Objective:**
This notebook provides a structured, reproducible exploratory analysis of the candidate video dataset prior to model training. It audits class balance, duration, frame rates, resolutions, decodability, and video-level split distributions to prevent data leakage and guarantee data integrity.

## 1. Dataset Overview
Initialize libraries, define filesystem paths, and establish standard visualization themes.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Establish plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Resolve project directories relative to notebook location
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"
MANIFEST_PATH = METADATA_DIR / "dataset_manifest.csv"
AUDIT_PATH = METADATA_DIR / "dataset_audit.csv"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Manifest Path: {MANIFEST_PATH}")

## 2. Dataset Provenance
Load and inspect the authoritative dataset manifest, checking licensing, source citations, and primary attributes.

In [ ]:
if not MANIFEST_PATH.exists():
    print(f"[INFO] Manifest file not yet populated at: {MANIFEST_PATH}")
    print("Please select a candidate dataset and run scripts/verify_dataset.py first.")
    df_manifest = None
else:
    df_manifest = pd.read_csv(MANIFEST_PATH)
    print(f"Total cataloged records: {len(df_manifest)}")
    display(df_manifest.head())
    print("\nUnique Sources:", df_manifest["source_dataset"].unique() if "source_dataset" in df_manifest else "N/A")
    print("Licenses:", df_manifest["license"].unique() if "license" in df_manifest else "N/A")

## 3. Class Distribution
Evaluate the proportion between `accident` and `normal` video samples to identify class imbalance.

In [ ]:
if df_manifest is not None and "class_label" in df_manifest.columns:
    class_counts = df_manifest["class_label"].value_counts()
    print("Class Counts:\n", class_counts)
    print("\nClass Proportions:\n", df_manifest["class_label"].value_counts(normalize=True) * 100)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#e74c3c", "#2ecc71"], ax=ax)
    ax.set_title("Distribution of Video Classes (Accident vs Normal)")
    ax.set_xlabel("Class Label")
    ax.set_ylabel("Number of Videos")
    plt.tight_layout()
    plt.show()
else:
    print("Class distribution analysis awaiting manifest population.")

## 4. Video Duration Analysis
Inspect the distribution of video clip durations to ensure temporal suitability for sliding window sequence extraction.

In [ ]:
if df_manifest is not None and "duration_seconds" in df_manifest.columns:
    print(df_manifest["duration_seconds"].describe())
    
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(data=df_manifest, x="duration_seconds", hue="class_label", kde=True, bins=20, ax=ax)
    ax.set_title("Video Duration Distribution by Class")
    ax.set_xlabel("Duration (seconds)")
    plt.tight_layout()
    plt.show()
else:
    print("Duration analysis awaiting manifest population.")

## 5. FPS Analysis
Analyze video frame rates to determine whether uniform frame rate normalization (e.g., standardizing to 25 or 30 FPS) is required during preprocessing.

In [ ]:
if df_manifest is not None and "fps" in df_manifest.columns:
    fps_counts = df_manifest["fps"].value_counts().sort_index()
    print("Frame Rate Distribution (FPS):\n", fps_counts)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.countplot(data=df_manifest, x="fps", palette="Blues_d", ax=ax)
    ax.set_title("Native Frame Rate (FPS) Frequency")
    plt.tight_layout()
    plt.show()
else:
    print("FPS analysis awaiting manifest population.")

## 6. Resolution Analysis
Verify spatial resolutions across clips to determine the optimal frame resizing and aspect ratio preservation strategy.

In [ ]:
if df_manifest is not None and "width" in df_manifest.columns and "height" in df_manifest.columns:
    df_manifest["resolution"] = df_manifest["width"].astype(str) + "x" + df_manifest["height"].astype(str)
    print("Top Spatial Resolutions:\n", df_manifest["resolution"].value_counts())
    
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.countplot(data=df_manifest, y="resolution", order=df_manifest["resolution"].value_counts().index, ax=ax)
    ax.set_title("Video Resolution Breakdown")
    plt.tight_layout()
    plt.show()
else:
    print("Resolution analysis awaiting manifest population.")

## 7. Corrupt Video Analysis
Verify stream integrity from `dataset_audit.csv` to confirm that all cataloged files are fully decodable by OpenCV.

In [ ]:
if AUDIT_PATH.exists():
    df_audit = pd.read_csv(AUDIT_PATH)
    corrupt_df = df_audit[df_audit["is_decodable"] == False]
    print(f"Total auditable videos : {len(df_audit)}")
    print(f"Corrupt / Unreadable   : {len(corrupt_df)}")
    if not corrupt_df.empty:
        print("\nCorrupted files identified:")
        display(corrupt_df[["filename", "notes"]])
    else:
        print("All audited video streams are valid and decodable.")
else:
    print("Audit log not found. Execute scripts/verify_dataset.py after selecting a dataset.")

## 8. Train Validation Test Distribution
Validate that the video-level partition holds zero video identifier overlap between splits and preserves class balance.

In [ ]:
if df_manifest is not None and "split" in df_manifest.columns:
    split_ct = pd.crosstab(df_manifest["split"], df_manifest["class_label"], margins=True)
    display(split_ct)
    
    # Validate zero video-level leakage
    train_ids = set(df_manifest[df_manifest["split"] == "TRAIN"]["video_id"])
    val_ids = set(df_manifest[df_manifest["split"] == "VALIDATION"]["video_id"])
    test_ids = set(df_manifest[df_manifest["split"] == "TEST"]["video_id"])
    
    overlap_tv = train_ids.intersection(val_ids)
    overlap_tt = train_ids.intersection(test_ids)
    overlap_vt = val_ids.intersection(test_ids)
    
    assert len(overlap_tv) == 0, f"Leakage between Train and Val: {overlap_tv}"
    assert len(overlap_tt) == 0, f"Leakage between Train and Test: {overlap_tt}"
    assert len(overlap_vt) == 0, f"Leakage between Val and Test: {overlap_vt}"
    print("\n[VERIFIED] Zero video ID leakage across Train, Validation, and Test splits.")
else:
    print("Split distribution awaiting execution of scripts/create_video_splits.py.")

## 9. Dataset Quality Checks
Perform automated boundary checks (detecting empty files, extreme duration outliers, and missing annotations).

In [ ]:
if df_manifest is not None:
    missing_fields = df_manifest.isnull().sum()
    print("Missing values per column:\n", missing_fields[missing_fields > 0])
    
    # Flag outlier durations (< 1.0s or > 60.0s)
    if "duration_seconds" in df_manifest.columns:
        outliers = df_manifest[(df_manifest["duration_seconds"] < 1.0) | (df_manifest["duration_seconds"] > 60.0)]
        print(f"\nDuration outliers (<1s or >60s): {len(outliers)}")
else:
    print("Quality checks awaiting manifest population.")

## 10. Findings & Next Steps
Document final observations once a dataset is selected and audited:
- **Class Balance Strategy:** (e.g., whether weighted loss or focal loss is indicated based on class ratio)
- **Frame Sampling Strategy:** (e.g., target sequence length $T=16$ at stride 1 or 2)
- **Spatial Standardization:** (target resize to $224 \times 224$)
- **Actionable Transition to Phase 3:** (proceeding with baseline model implementation once audit checks pass)